# Challenge 7: Judge calibration

**EDD: Measure. Model upgrade framework, step 4 (trust), second half: a judge may only block a release if it agrees with the people whose judgement you trust.**
You'll pull reviewed examples with the SDK, measure judge-human agreement (accuracy and Cohen's kappa), read the disagreements, retune the judge, and measure again.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import json
import os
import sys
import warnings


for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client, uuid7

import lab_links
import lab_sdk

client = Client()
PROD = lab_sdk.prod_project()
GOLDEN = lab_sdk.ds_golden()
print("prod project:", PROD)
print("golden dataset:", GOLDEN)

from collections import Counter
from concurrent.futures import ThreadPoolExecutor

from eval_lab.evaluators import judge_json
QUEUE = lab_sdk.queue_name()

## 1. Reviewer labels

In real use these come from annotation queues or the UI. To keep this track self-contained, this cell writes
`human_correct` labels (1 = the agent handled it correctly) with a reviewer comment onto 14 runs. They are seeded
stand-ins for reviewers, not real human judgements.

In [ ]:
labelled = [r for r in lab_sdk.roots(client) if not r.error][:14]
for r in labelled:
    score, comment = lab_sdk.human_label(r)
    client.create_feedback(run_id=r.id, session_id=r.session_id, key="human_correct", score=score, comment=comment,
                           source_info={"reviewer": "qa-team", "channel": "seeded-stand-in"})
client.flush()
seen = lab_sdk.wait_for_feedback(client, "human_correct", [r.id for r in labelled], expected=len(labelled))
print(f"human_correct on {len(seen)} of {len(labelled)} runs")

## 2. Extract reviewed examples

`client.list_feedback(run_ids=..., feedback_key=[...])` returns the labels; join them to their runs.

In [ ]:
by_id = {r.id: r for r in labelled}
reviewed = []
for fb in client.list_feedback(run_ids=list(by_id), feedback_key=["human_correct"]):
    r = by_id[fb.run_id]
    reviewed.append({"run": r, "human": int(fb.score), "human_comment": fb.comment})
print(len(reviewed), "reviewed examples;", sum(1 for x in reviewed if x["human"] == 0), "labelled incorrect")

## 3. Judge v1

The first judge prompt asks a general question. Run it on every reviewed example and store its verdict as `judge_v1`.

In [ ]:
JUDGE_V1 = (
    "Customer asked: {query}\nTools the assistant called: {tools}\nAssistant answered: {answer}\n"
    "Is this a good response? Reply with only JSON: {{\"correct\": 0 or 1, \"reason\": \"one sentence\"}}"
)


def run_judge(template, key):
    def one(x):
        r = x["run"]
        verdict = judge_json(template.format(
            query=lab_sdk.redact(str(r.inputs.get("query"))),
            tools=(r.outputs or {}).get("trajectory"),
            answer=lab_sdk.redact(str((r.outputs or {}).get("answer")))))
        client.create_feedback(run_id=r.id, session_id=r.session_id, key=key, score=verdict.get("correct"), comment=verdict.get("reason"),
                               feedback_source_type="model")
        x[key] = verdict.get("correct")
        x[key + "_reason"] = verdict.get("reason")
    with ThreadPoolExecutor(max_workers=4) as pool:
        list(pool.map(one, reviewed))
    client.flush()


run_judge(JUDGE_V1, "judge_v1")
print("judge_v1 scored", sum(1 for x in reviewed if x["judge_v1"] is not None), "of", len(reviewed))

## 4. Measure agreement

Accuracy alone flatters a judge that says "1" to everything, so also compute **Cohen's kappa**, which discounts agreement
expected by chance. Write `agreement(human, judge)` returning `{"accuracy", "kappa", "n"}`. A reference solution is in the next cell.

In [ ]:
# Reference solution: try writing this yourself first.
def agreement(human, judge):
    n = len(human)
    observed = sum(h == j for h, j in zip(human, judge)) / n
    p_h, p_j = sum(human) / n, sum(judge) / n
    expected = p_h * p_j + (1 - p_h) * (1 - p_j)
    kappa = 1.0 if expected == 1 else (observed - expected) / (1 - expected)
    return {"accuracy": round(observed, 3), "kappa": round(kappa, 3), "n": n}

In [ ]:
def pairs(key):
    usable = [x for x in reviewed if x.get(key) is not None]
    return [x["human"] for x in usable], [int(x[key]) for x in usable]


report_v1 = agreement(*pairs("judge_v1"))
print("judge v1 vs humans:", report_v1)

## 5. Adjudicate the differences

Read each disagreement. Is the judge wrong, or is the label? Here the reviewers are right: the process rule is that a refund
is only allowed after an eligibility check, and v1 never mentions it.

In [ ]:
for x in reviewed:
    if x.get("judge_v1") is not None and int(x["judge_v1"]) != x["human"]:
        print("query  :", lab_sdk.redact(str(x["run"].inputs.get("query")))[:70])
        print("tools  :", (x["run"].outputs or {}).get("trajectory"))
        print("human  :", x["human"], "-", x["human_comment"])
        print("judge  :", x["judge_v1"], "-", x["judge_v1_reason"])
        print()

## 6. Retune and re-measure

Put the rule the reviewers apply into the judge prompt and score again. The same loop is what **Align Evals** automates in the
LangSmith UI: an evaluator is aligned to labelled examples and you iterate on its prompt until the scores match.

In [ ]:
JUDGE_V2 = (
    "Customer asked: {query}\nTools the assistant called: {tools}\nAssistant answered: {answer}\n"
    "Policy: a refund may only be issued after check_refund_eligibility was called for that order. "
    "If issue_refund was called without check_refund_eligibility, the response is incorrect (0), even if the order "
    "turns out to be eligible. If the assistant gave no answer, it is incorrect (0). Otherwise judge whether the "
    "answer is accurate. Reply with only JSON: {{\"correct\": 0 or 1, \"reason\": \"one sentence\"}}"
)
run_judge(JUDGE_V2, "judge_v2")
report_v2 = agreement(*pairs("judge_v2"))
print("judge v1:", report_v1)
print("judge v2:", report_v2)

os.makedirs(lab_sdk.EXPORT_DIR, exist_ok=True)
with open(f"{lab_sdk.EXPORT_DIR}/calibration.json", "w") as f:
    json.dump({"accuracy_v1": report_v1["accuracy"], "kappa_v1": report_v1["kappa"],
               "accuracy_v2": report_v2["accuracy"], "kappa_v2": report_v2["kappa"], "n": report_v2["n"]}, f, indent=2)

## 7. See it in LangSmith

In [ ]:
lab_links.queue_link(QUEUE, hint="reviewers label runs here; their labels are what the judge is measured against")
lab_links.project_link(PROD, hint="filter runs with feedback human_correct, judge_v1, judge_v2 and compare them")

---

# Optional: Align Evals in the UI

1. In LangSmith, open **Evaluators**, create an LLM-as-judge evaluator, and choose **Align** (Align Evals).
2. Select the `human_correct` labelled runs as the alignment set.
3. Edit the evaluator prompt until the alignment score stops improving.

This automates the retune loop you just did by hand. It depends on your LangSmith plan.

